# Task 3 — XGBoost pipeline (Monthly Spending, NMAE)
NMAE = 1 / (1 + Σ|y−ŷ| / Σy) → ลด **ผลรวม absolute error** = ใช้ objective `reg:absoluteerror`
เทียบ 3 แบบด้วย 5-fold OOF: (0) ทาย 0 ทุกแถว = 0.5 · (A) XGB MAE ตรง ๆ · (B) two-stage: จะจ่ายไหม → จ่ายเท่าไร
**ไม่ submit** — เขียนไฟล์ลง `/kaggle/working/` เท่านั้น · ตั้ง Accelerator = GPU (ถ้ามี) หรือ CPU ก็ได้

In [ ]:
import os, time, warnings, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")
SEED, NF = 42, 5
DEVICE = "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"
print("xgboost", xgb.__version__, "| device", DEVICE)

def walk_find(name, must):
    for d, _, fs in os.walk("/kaggle/input", followlinks=True):
        if name in fs and must in d: return os.path.join(d, name)
TRAIN = walk_find("train.csv", "03_regression"); TEST = walk_find("test.csv", "03_regression")
SAMPLE = next((os.path.join(d, "sample_submission.csv") for d, _, fs in os.walk("/kaggle/input", followlinks=True) if "sample_submission.csv" in fs), None)
print(TRAIN, TEST, SAMPLE, sep="\n")
tr, te = pd.read_csv(TRAIN), pd.read_csv(TEST)
TARGET = "spending_30d"; y = tr[TARGET].values
print(tr.shape, te.shape, "| y=0:", round((y == 0).mean(), 3), "| mean", round(y.mean(), 1), "| max", y.max())

## 1. Features (raw + ratio ง่าย ๆ)

In [ ]:
def add_feats(d):
    d = d.copy(); eps = 1.0
    d["hist_per_day"]      = d["historical_spending"] / (d["account_age_days"] + eps)
    d["tx_per_day"]        = d["total_transactions"] / (d["account_age_days"] + eps)
    d["exp_monthly_spend"] = d["avg_transaction_value"] * d["purchase_frequency"]
    d["prev_vs_hist"]      = d["prev_month_spending"] / (d["historical_spending"] + eps)
    d["recency_ratio"]     = d["days_since_last_purchase"] / (d["days_since_last_login"] + eps)
    d["play_per_session"]  = d["total_playtime_hours"] / (d["sessions_per_week"] + eps)
    return d
FEATS = [c for c in add_feats(te).columns if c not in ("player_id", TARGET)]
X, Xt = add_feats(tr)[FEATS], add_feats(te)[FEATS]
print(len(FEATS), "features")

## 2. 5-fold CV

In [ ]:
def nmae(yt, yp): return 1 / (1 + np.abs(yt - yp).sum() / yt.sum())
P_REG = dict(objective="reg:absoluteerror", n_estimators=4000, learning_rate=0.03, max_depth=7, min_child_weight=5,
             subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist", device=DEVICE,
             early_stopping_rounds=150, random_state=SEED)
P_CLF = dict(objective="binary:logistic", eval_metric="logloss", n_estimators=3000, learning_rate=0.03, max_depth=6,
             min_child_weight=5, subsample=0.8, colsample_bytree=0.8, tree_method="hist", device=DEVICE,
             early_stopping_rounds=150, random_state=SEED)

folds = list(StratifiedKFold(NF, shuffle=True, random_state=SEED).split(X, (y > 0).astype(int)))
oof_a, oof_p, oof_r = np.zeros(len(y)), np.zeros(len(y)), np.zeros(len(y))
te_a, te_p, te_r = np.zeros(len(Xt)), np.zeros(len(Xt)), np.zeros(len(Xt))
imp = pd.Series(0.0, index=FEATS)
for k, (a, b) in enumerate(folds):
    t0 = time.time()
    m = xgb.XGBRegressor(**P_REG).fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])], verbose=False)       # (A)
    oof_a[b] = m.predict(X.iloc[b]); te_a += m.predict(Xt) / NF
    imp += pd.Series(m.feature_importances_, index=FEATS) / NF
    c = xgb.XGBClassifier(**P_CLF).fit(X.iloc[a], (y[a] > 0).astype(int), eval_set=[(X.iloc[b], (y[b] > 0).astype(int))], verbose=False)
    oof_p[b] = c.predict_proba(X.iloc[b])[:, 1]; te_p += c.predict_proba(Xt)[:, 1] / NF                      # (B) stage 1
    pa, pb = a[y[a] > 0], b[y[b] > 0]
    r = xgb.XGBRegressor(**P_REG).fit(X.iloc[pa], y[pa], eval_set=[(X.iloc[pb], y[pb])], verbose=False)     # (B) stage 2
    oof_r[b] = r.predict(X.iloc[b]); te_r += r.predict(Xt) / NF
    print(f"fold {k}: A {nmae(y[b], np.clip(oof_a[b], 0, None)):.4f} | iters A {m.best_iteration} clf {c.best_iteration} reg+ {r.best_iteration} | {time.time()-t0:.0f}s")

## 3. เลือกแบบที่ดีที่สุด

In [ ]:
res = {"(0) all zero": nmae(y, np.zeros_like(y)), "(A) XGB MAE": nmae(y, np.clip(oof_a, 0, None))}
ths = np.arange(0.1, 0.95, 0.025)
sc = [nmae(y, np.where(oof_p >= t, np.clip(oof_r, 0, None), 0)) for t in ths]
BEST_T = float(ths[int(np.argmax(sc))]); res[f"(B) two-stage @p>={BEST_T:.3f}"] = max(sc)
for k_, v in res.items(): print(f"{k_:28s} NMAE {v:.4f}")
USE = "B" if max(sc) > res["(A) XGB MAE"] else "A"
pred = np.where(te_p >= BEST_T, np.clip(te_r, 0, None), 0) if USE == "B" else np.clip(te_a, 0, None)
print("\nใช้แบบ", USE, "| test: ทาย 0 =", round((pred == 0).mean(), 3), "| mean", round(pred.mean(), 1))
print("\nTop 15 features (A):"); print(imp.sort_values(ascending=False).head(15).round(4).to_string())

## 4. เขียนไฟล์ (ไม่ submit)

In [ ]:
os.makedirs("/kaggle/working/preds", exist_ok=True)
out = pd.DataFrame({"row": np.arange(len(pred)), "pred": np.round(pred, 2)})
if SAMPLE:
    ids = pd.read_csv(SAMPLE)["id"]; assert len(ids) == len(pred); out.insert(0, "id", ids.values)   # จับคู่ตามลำดับแถว
out.to_csv("/kaggle/working/preds/t3_xgb.csv", index=False)
np.save("/kaggle/working/preds/t3_xgb_oof.npy", np.c_[oof_a, oof_p, oof_r])
print(out.head())